# WF2020 Low-Exposure Floor Representation Drift v1

Purpose: explain why the training-safe low-exposure rally floor target worsened `wf_2020` despite the post-hoc overlay screen passing. This is diagnostic evidence only; it does not promote an actor.

Baselines:
- Canonical: `v19reg_v22_budget_usage_bull_guard_v1`
- Direct comparison: `low_exposure_rally_floor_three_fold_checkpoint_metric_replay_v1`
- Prior better replay: `return_preserving_bull_participation_three_fold_checkpoint_metric_replay_v1`

Hypothesis: the low-exposure floor changed too many broad rebound labels, diluted the original high-confidence rebound slice, and increased mid-year exposure without fixing the late-2020 low-exposure rally state. The result is representation drift, not a simple reward/floor insufficiency.


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'runtime').exists():
            return candidate
    raise RuntimeError('repo root not found')

ROOT=find_repo_root(Path.cwd())
EXPERIMENT='wf2020_low_exposure_floor_representation_drift_v1'
CARD=ROOT/'runtime/model_selection/card'/EXPERIMENT
CARD.mkdir(parents=True,exist_ok=True)
RETURN_REPLAY=ROOT/'runtime/model_selection/card/return_preserving_bull_participation_three_fold_checkpoint_metric_replay_v1'
LOW_REPLAY=ROOT/'runtime/model_selection/card/low_exposure_rally_floor_three_fold_checkpoint_metric_replay_v1'
RETURN_TARGET=ROOT/'runtime/model_selection/card/wf2020_return_preserving_bull_participation_target_v1/return_preserving_bull_participation_three_fold_target_contract.csv'
LOW_TARGET=ROOT/'runtime/model_selection/card/wf2020_low_exposure_rally_floor_target_v1/low_exposure_rally_floor_three_fold_target_contract.csv'
LOW_ATTRIBUTION=ROOT/'runtime/model_selection/card/wf2020_failed_fold_rally_gap_attribution_v1/summary.json'

SIDE_CARS={
 'monthly_replay_delta': CARD/'monthly_replay_delta.csv',
 'slice_metric_delta': CARD/'slice_metric_delta.csv',
 'target_distribution_delta': CARD/'target_distribution_delta.csv',
 'late_rally_path_delta': CARD/'late_rally_path_delta.csv',
 'selection': CARD/'selection.json',
 'summary': CARD/'summary.json',
 'conclusion': CARD/'conclusion.md',
}

def rel(path: Path)->str:
    return str(path.relative_to(ROOT))

def json_safe(value):
    if isinstance(value,dict): return {str(k):json_safe(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)): return [json_safe(v) for v in value]
    if isinstance(value,(np.integer,)): return int(value)
    if isinstance(value,(np.bool_,)): return bool(value)
    if isinstance(value,(np.floating,)): value=float(value)
    if isinstance(value,float): return value if math.isfinite(value) else None
    if isinstance(value,pd.Timestamp): return value.isoformat()
    if isinstance(value,Path): return str(value)
    return value

def write_json(path: Path,payload:dict)->None:
    path.write_text(json.dumps(json_safe(payload),indent=2,sort_keys=True,allow_nan=False)+'\n',encoding='utf-8')

def return_pct(series: pd.Series)->float:
    return float((series.iloc[-1]/series.iloc[0]-1)*100.0)

def max_drawdown_pct(series: pd.Series)->float:
    return float(((series.cummax()-series)/series.cummax()).max()*100.0)

print('card', rel(CARD))


card runtime/model_selection/card/wf2020_low_exposure_floor_representation_drift_v1


In [2]:
return_summary=json.loads((RETURN_REPLAY/'summary.json').read_text())
low_summary=json.loads((LOW_REPLAY/'summary.json').read_text())
attr_summary=json.loads(LOW_ATTRIBUTION.read_text())
return_sel=return_summary['selected_by_fold']['wf_2020']
low_sel=low_summary['selected_by_fold']['wf_2020']
print('return preserving selected', return_sel)
print('low floor selected', low_sel)


return preserving selected {'bc_max_dd': 17.27796837214841, 'bc_relative_return': -275.6146761274102, 'candidate_count': 4, 'fold': 'wf_2020', 'selected_candidate': 'bc_initialization', 'selected_candidate_gate': False, 'selected_daily_path': 'runtime/model_selection/card/return_preserving_bull_participation_three_fold_checkpoint_metric_replay_v1/official_daily_paths/wf_2020__bc_initialization_official_daily_path.csv', 'selected_drift_gate': True, 'selected_iter': 0, 'selected_kind': 'bc_initialization', 'selected_loss': 0.045068572046479, 'selected_max_dd': 17.27796837214841, 'selected_relative_return': -275.6146761274102, 'selected_target_gate': False, 'status': 'evaluated'}
low floor selected {'bc_max_dd': 20.38458883710407, 'bc_relative_return': -278.93465341661, 'candidate_count': 4, 'fold': 'wf_2020', 'selected_candidate': 'bc_initialization', 'selected_candidate_gate': False, 'selected_daily_path': 'runtime/model_selection/card/low_exposure_rally_floor_three_fold_checkpoint_metr

In [3]:
def load_daily(replay: Path, selected: dict, label: str)->pd.DataFrame:
    df=pd.read_csv(ROOT/selected['selected_daily_path'],parse_dates=['datetime']).sort_values('datetime').reset_index(drop=True)
    df['source']=label
    df['month']=df.datetime.dt.to_period('M').astype(str)
    df['strategy_ret']=df.strategy_nav.pct_change().fillna(0.0)
    df['price_ret']=df.price.pct_change().fillna(0.0)
    df['drawdown_pct']=(df.strategy_nav.cummax()-df.strategy_nav)/df.strategy_nav.cummax()*100.0
    return df

ret_daily=load_daily(RETURN_REPLAY, return_sel, 'return_preserving')
low_daily=load_daily(LOW_REPLAY, low_sel, 'low_exposure_floor')
rows=[]
for label,df in [('return_preserving',ret_daily),('low_exposure_floor',low_daily)]:
    for month,g in df.groupby('month'):
        rows.append({
            'source': label,
            'month': month,
            'rows': int(len(g)),
            'strategy_return_pct': return_pct(g.strategy_nav),
            'price_return_pct': return_pct(g.price),
            'avg_position': float(g.position.mean()),
            'min_position': float(g.position.min()),
            'max_position': float(g.position.max()),
            'buy_days': int((g.action.astype(str).str.upper()=='BUY').sum()),
            'sell_days': int((g.action.astype(str).str.upper()=='SELL').sum()),
            'max_drawdown_pct_in_month': float(g.drawdown_pct.max()),
        })
monthly=pd.DataFrame(rows)
wide=monthly.pivot(index='month',columns='source',values=['strategy_return_pct','avg_position','max_position','buy_days','sell_days','max_drawdown_pct_in_month']).reset_index()
wide.columns=['_'.join([str(x) for x in col if str(x)]) for col in wide.columns.to_flat_index()]
for metric in ['strategy_return_pct','avg_position','max_position','buy_days','sell_days','max_drawdown_pct_in_month']:
    wide[f'{metric}_delta_low_minus_return']=wide[f'{metric}_low_exposure_floor']-wide[f'{metric}_return_preserving']
wide.to_csv(SIDE_CARS['monthly_replay_delta'],index=False)
display(wide)


,month,strategy_return_pct_low_exposure_floor,strategy_return_pct_return_preserving,avg_position_low_exposure_floor,avg_position_return_preserving,max_position_low_exposure_floor,max_position_return_preserving,buy_days_low_exposure_floor,buy_days_return_preserving,sell_days_low_exposure_floor,sell_days_return_preserving,max_drawdown_pct_in_month_low_exposure_floor,max_drawdown_pct_in_month_return_preserving,strategy_return_pct_delta_low_minus_return,avg_position_delta_low_minus_return,max_position_delta_low_minus_return,buy_days_delta_low_minus_return,sell_days_delta_low_minus_return,max_drawdown_pct_in_month_delta_low_minus_return
0,2020-01,0.000000,0.000000,0.110060,0.077633,0.255424,0.129529,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.032427,0.125894,0.0,0.0,0.000000
1,2020-02,0.000000,0.000000,0.120238,0.120898,0.162500,0.162500,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,-0.000660,0.000000,0.0,0.0,0.000000
2,2020-03,-0.023369,-0.011685,0.362503,0.263015,0.660042,0.667428,1.0,1.0,0.0,0.0,1.587954,0.800239,-0.011685,0.099488,-0.007387,0.0,0.0,0.787715
3,2020-04,10.150147,5.470425,0.364538,0.364989,0.635642,0.643272,0.0,0.0,2.0,2.0,5.310176,2.676027,4.679722,-0.000452,-0.007630,0.0,0.0,2.634149
4,2020-05,3.046551,3.080743,0.185587,0.186156,0.195676,0.195942,1.0,1.0,1.0,1.0,2.542775,2.555754,-0.034193,-0.000569,-0.000266,0.0,0.0,-0.012979
5,2020-06,8.501833,3.647861,0.530883,0.391720,0.700000,0.603959,2.0,1.0,2.0,3.0,6.696584,7.674283,4.853973,0.139163,0.096041,1.0,-1.0,-0.977699
6,2020-07,8.705268,8.706474,0.382530,0.225409,0.575000,0.461071,0.0,0.0,3.0,3.0,2.720477,2.720919,-0.001206,0.157121,0.113929,0.0,0.0,-0.000442
7,2020-08,4.045963,4.049732,0.413596,0.346162,0.583950,0.545625,1.0,1.0,1.0,1.0,2.207980,2.208965,-0.003768,0.067434,0.038325,0.0,0.0,-0.000986
8,2020-09,-6.314801,-2.986357,0.264866,0.303946,0.457345,0.476641,2.0,1.0,2.0,3.0,20.384589,17.277968,-3.328445,-0.039080,-0.019296,1.0,-1.0,3.106620
9,2020-10,-1.848117,-1.864109,0.097810,0.097627,0.097810,0.097627,1.0,2.0,3.0,3.0,12.160993,8.356649,0.015992,0.000182,0.000182,-1.0,0.0,3.804344


In [4]:
def load_slices(replay: Path, label: str)->pd.DataFrame:
    df=pd.read_csv(replay/'bc_slice_metrics.csv')
    df=df[df.fold.eq('wf_2020')].copy()
    df['source']=label
    return df
slices=pd.concat([load_slices(RETURN_REPLAY,'return_preserving'),load_slices(LOW_REPLAY,'low_exposure_floor')],ignore_index=True)
left=slices[slices.source.eq('low_exposure_floor')].merge(
    slices[slices.source.eq('return_preserving')],
    on=['fold','slice'],suffixes=('_low','_return'),how='outer'
)
for col in ['samples','accuracy','loss','label_unique_actions','predicted_unique_actions','label_dominant_action','label_dominant_share','predicted_dominant_action','predicted_dominant_share']:
    if f'{col}_low' in left.columns and f'{col}_return' in left.columns:
        left[f'{col}_delta_low_minus_return']=pd.to_numeric(left[f'{col}_low'],errors='coerce')-pd.to_numeric(left[f'{col}_return'],errors='coerce')
left.to_csv(SIDE_CARS['slice_metric_delta'],index=False)
display(left)


,fold,slice,samples_low,accuracy_low,loss_low,label_unique_actions_low,predicted_unique_actions_low,label_dominant_action_low,label_dominant_share_low,predicted_dominant_action_low,...,source_return,samples_delta_low_minus_return,accuracy_delta_low_minus_return,loss_delta_low_minus_return,label_unique_actions_delta_low_minus_return,predicted_unique_actions_delta_low_minus_return,label_dominant_action_delta_low_minus_return,label_dominant_share_delta_low_minus_return,predicted_dominant_action_delta_low_minus_return,predicted_dominant_share_delta_low_minus_return
0,wf_2020,all,1905,0.772178,1.138416,5,4,1,0.275591,4,...,return_preserving,0,0.006299,-0.004102,0,0,0,-0.020997,0,-0.057218
1,wf_2020,drawdown_control_state,715,0.889510,1.020240,5,4,1,0.573427,1,...,return_preserving,0,-0.011189,0.008745,0,0,0,0.000000,0,0.001399
2,wf_2020,is_downtrend_state,715,0.889510,1.020240,5,4,1,0.573427,1,...,return_preserving,0,-0.011189,0.008745,0,0,0,0.000000,0,0.001399
3,wf_2020,is_rebound_cluster,550,0.883636,1.031991,5,4,2,0.421818,2,...,return_preserving,358,0.055511,-0.049225,2,2,-2,-0.406307,-2,-0.440701
4,wf_2020,release_gate_state,32,0.937500,0.974425,5,4,4,0.343750,2,...,return_preserving,0,0.000000,-0.000734,0,0,0,0.000000,-2,-0.031250


In [5]:
def bool_series(frame: pd.DataFrame,col: str)->pd.Series:
    if col not in frame.columns:
        return pd.Series(False,index=frame.index)
    s=frame[col]
    if s.dtype==bool:
        return s.fillna(False)
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(float).ne(0)
    return s.fillna('').astype(str).str.lower().str.strip().isin({'1','true','yes','y'})

def load_target(path: Path, label: str)->pd.DataFrame:
    df=pd.read_csv(path,parse_dates=['datetime','train_start','train_end'])
    df=df[df.fold.eq('wf_2020_train') & df.datetime.between(df.train_start,df.train_end)].copy()
    df['source']=label
    return df
ret_target=load_target(RETURN_TARGET,'return_preserving')
low_target=load_target(LOW_TARGET,'low_exposure_floor')
rows=[]
for label,df in [('return_preserving',ret_target),('low_exposure_floor',low_target)]:
    target_col='return_preserving_target_position' if 'return_preserving_target_position' in df.columns else 'consumed_state_objective_target_position'
    if target_col not in df.columns:
        target_col='risk_target_position'
    for segment,mask in {
        'all': pd.Series(True,index=df.index),
        'rebound': bool_series(df,'is_rebound_cluster'),
        'downtrend': bool_series(df,'is_downtrend_state'),
        'low_exposure_floor': bool_series(df,'low_exposure_rally_floor_state_v1'),
        'non_downtrend_low_target': (~bool_series(df,'is_downtrend_state')) & (pd.to_numeric(df[target_col],errors='coerce')<0.35),
    }.items():
        g=df[mask].copy()
        if g.empty:
            rows.append({'source':label,'segment':segment,'rows':0})
            continue
        target=pd.to_numeric(g[target_col],errors='coerce')
        rows.append({
            'source':label,
            'segment':segment,
            'rows':int(len(g)),
            'mean_target':float(target.mean()),
            'p10_target':float(target.quantile(.1)),
            'p50_target':float(target.quantile(.5)),
            'p90_target':float(target.quantile(.9)),
            'mean_objective_weight':float(pd.to_numeric(g.get('objective_weight',1.0),errors='coerce').fillna(1.0).mean()),
            'rebound_rows':int(bool_series(g,'is_rebound_cluster').sum()),
            'downtrend_rows':int(bool_series(g,'is_downtrend_state').sum()),
        })
dist=pd.DataFrame(rows)
pivot=dist[dist.source.eq('low_exposure_floor')].merge(dist[dist.source.eq('return_preserving')],on='segment',suffixes=('_low','_return'),how='outer')
for col in ['rows','mean_target','p10_target','p50_target','p90_target','mean_objective_weight','rebound_rows','downtrend_rows']:
    if f'{col}_low' in pivot and f'{col}_return' in pivot:
        pivot[f'{col}_delta_low_minus_return']=pd.to_numeric(pivot[f'{col}_low'],errors='coerce')-pd.to_numeric(pivot[f'{col}_return'],errors='coerce')
pivot.to_csv(SIDE_CARS['target_distribution_delta'],index=False)
display(pivot)


,source_low,segment,rows_low,mean_target_low,p10_target_low,p50_target_low,p90_target_low,mean_objective_weight_low,rebound_rows_low,downtrend_rows_low,...,rebound_rows_return,downtrend_rows_return,rows_delta_low_minus_return,mean_target_delta_low_minus_return,p10_target_delta_low_minus_return,p50_target_delta_low_minus_return,p90_target_delta_low_minus_return,mean_objective_weight_delta_low_minus_return,rebound_rows_delta_low_minus_return,downtrend_rows_delta_low_minus_return
0,low_exposure_floor,all,2470,0.335576,0.1500,0.30,0.65,7.434201,737.0,897.0,...,285.0,897.0,0,0.030976,0.0621,0.1453,0.0000,1.097976,452.0,0.0
1,low_exposure_floor,downtrend,897,0.181360,0.1500,0.15,0.25,14.144344,7.0,897.0,...,7.0,897.0,0,0.000000,0.0000,0.0000,0.0000,0.000000,0.0,0.0
2,low_exposure_floor,low_exposure_floor,452,0.300000,0.3000,0.30,0.30,7.000000,452.0,0.0,...,NaN,NaN,452,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,low_exposure_floor,non_downtrend_low_target,693,0.252802,0.0867,0.30,0.30,4.913420,452.0,0.0,...,0.0,0.0,0,0.110405,0.0278,0.1858,0.0471,3.913420,452.0,0.0
4,low_exposure_floor,rebound,737,0.444318,0.3000,0.30,0.70,6.709491,737.0,7.0,...,285.0,7.0,452,-0.228884,-0.3500,-0.3500,0.0000,0.460737,452.0,0.0


In [6]:
late=ret_daily[['datetime','price','strategy_nav','position','action']].merge(
    low_daily[['datetime','strategy_nav','position','action']],
    on='datetime',suffixes=('_return','_low')
)
late=late[late.datetime>=pd.Timestamp('2020-10-01')].copy()
late['position_delta_low_minus_return']=late['position_low']-late['position_return']
late['strategy_nav_delta_low_minus_return']=late['strategy_nav_low']-late['strategy_nav_return']
late['action_changed']=late['action_low'].astype(str).ne(late['action_return'].astype(str))
late.to_csv(SIDE_CARS['late_rally_path_delta'],index=False)
print('late rows',len(late),'action_changed',int(late.action_changed.sum()),'mean_pos_delta',float(late.position_delta_low_minus_return.mean()))
display(late.head(20))


late rows 63 action_changed 14 mean_pos_delta 0.00018219920050660532


,datetime,price,strategy_nav_return,position_return,action_return,strategy_nav_low,position_low,action_low,position_delta_low_minus_return,strategy_nav_delta_low_minus_return,action_changed
189,2020-10-01,17.927334,1.258991e+06,0.097627,SELL,1.307547e+06,0.09781,HOLD,0.000182,48556.327595,True
190,2020-10-02,16.329332,1.236548e+06,0.097627,SELL,1.252534e+06,0.09781,SELL,0.000182,15986.103804,False
191,2020-10-05,17.923332,1.258545e+06,0.097627,HOLD,1.307368e+06,0.09781,HOLD,0.000182,48822.500975,False
192,2020-10-06,17.617332,1.254322e+06,0.097627,HOLD,1.296841e+06,0.09781,HOLD,0.000182,42518.906004,False
193,2020-10-07,18.680000,1.268987e+06,0.097627,HOLD,1.333397e+06,0.09781,HOLD,0.000182,64409.863646,False
194,2020-10-08,19.328667,1.277939e+06,0.097627,HOLD,1.355711e+06,0.09781,HOLD,0.000182,77772.391112,False
195,2020-10-09,20.342667,1.291932e+06,0.097627,HOLD,1.390593e+06,0.09781,HOLD,0.000182,98660.789854,False
196,2020-10-12,21.226000,1.303764e+06,0.097627,SELL,1.419960e+06,0.09781,SELL,0.000182,116196.670094,False
197,2020-10-13,21.200666,1.303609e+06,0.097627,HOLD,1.419644e+06,0.09781,HOLD,0.000182,116034.536305,False
198,2020-10-14,20.864668,1.301559e+06,0.097627,HOLD,1.415444e+06,0.09781,HOLD,0.000182,113884.145680,False


In [7]:
base_ret=float(return_sel['selected_relative_return'])
low_ret=float(low_sel['selected_relative_return'])
base_dd=float(return_sel['selected_max_dd'])
low_dd=float(low_sel['selected_max_dd'])
monthly_wide=pd.read_csv(SIDE_CARS['monthly_replay_delta'])
sep=monthly_wide[monthly_wide.month.eq('2020-09')].iloc[0].to_dict()
nov=monthly_wide[monthly_wide.month.eq('2020-11')].iloc[0].to_dict()
dec=monthly_wide[monthly_wide.month.eq('2020-12')].iloc[0].to_dict()
slice_delta=pd.read_csv(SIDE_CARS['slice_metric_delta'])
rebound_row=slice_delta[slice_delta['slice'].eq('is_rebound_cluster')].iloc[0].to_dict()
target_delta=pd.read_csv(SIDE_CARS['target_distribution_delta'])
rebound_target=target_delta[target_delta['segment'].eq('rebound')].iloc[0].to_dict()

selection={
 'experiment': EXPERIMENT,
 'created_at': datetime.now(timezone.utc).isoformat(),
 'decision': 'low_exposure_floor_representation_drift_confirmed_no_promotion',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'low_exposure_rally_floor_three_fold_checkpoint_metric_replay_v1',
 'reference_baseline': 'soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05',
 'role': 'diagnose',
 'changed_layer': ['model_selection','training_target_composition','representation'],
 'target_window': 'wf_2020',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'training_allowed_from_this_card': False,
 'relative_return_delta_low_minus_return_pp': low_ret-base_ret,
 'max_dd_delta_low_minus_return_pp': low_dd-base_dd,
 'september_return_delta_pp': float(sep['strategy_return_pct_delta_low_minus_return']),
 'november_return_delta_pp': float(nov['strategy_return_pct_delta_low_minus_return']),
 'december_return_delta_pp': float(dec['strategy_return_pct_delta_low_minus_return']),
 'rebound_slice_sample_delta': int(rebound_row['samples_delta_low_minus_return']),
 'rebound_slice_label_dominant_action_low': int(rebound_row['label_dominant_action_low']),
 'rebound_slice_label_dominant_action_return': int(rebound_row['label_dominant_action_return']),
 'rebound_target_rows_delta': int(rebound_target['rows_delta_low_minus_return']),
 'late_rally_mean_position_delta': float(late['position_delta_low_minus_return'].mean()),
 'late_rally_action_changed_rows': int(late['action_changed'].sum()),
 'next_mechanism': 'discover_narrow_late_rally_state_feature_or_reward_before_more_target_revision',
 'sidecars': [path.name for path in SIDE_CARS.values()],
}
summary=dict(selection)
summary.update({
 'return_preserving_replay_summary': rel(RETURN_REPLAY/'summary.json'),
 'low_floor_replay_summary': rel(LOW_REPLAY/'summary.json'),
 'source_low_exposure_attribution': rel(LOW_ATTRIBUTION),
 'conclusion': [
    f'Low-exposure floor replay worsened wf_2020 relative return by {low_ret-base_ret:.2f}pp and MaxDD by {low_dd-base_dd:.2f}pp.',
    f'Rebound slice samples expanded by {int(rebound_row["samples_delta_low_minus_return"])} and dominant label shifted from action {int(rebound_row["label_dominant_action_return"])} to action {int(rebound_row["label_dominant_action_low"])}.',
    f'Late-rally mean position barely changed ({float(late["position_delta_low_minus_return"].mean()):.6f}), while September/December returns deteriorated.',
    'The next step should discover a narrower late-rally state feature or reward gate before another target revision.',
 ],
})
write_json(SIDE_CARS['selection'],selection)
write_json(SIDE_CARS['summary'],summary)
lines=[
 f'# {EXPERIMENT} conclusion','',f"Decision: `{selection['decision']}`.",'',
 'Key evidence:',
 f'- Low-exposure replay relative return delta vs prior: `{low_ret-base_ret:.2f}pp`; MaxDD delta: `{low_dd-base_dd:.2f}pp`.',
 f'- Rebound slice samples expanded by `{int(rebound_row["samples_delta_low_minus_return"])}`; dominant label shifted from action `{int(rebound_row["label_dominant_action_return"])}` to action `{int(rebound_row["label_dominant_action_low"])}`.',
 f'- September strategy return delta: `{float(sep["strategy_return_pct_delta_low_minus_return"]):.2f}pp`; November delta: `{float(nov["strategy_return_pct_delta_low_minus_return"]):.2f}pp`; December delta: `{float(dec["strategy_return_pct_delta_low_minus_return"]):.2f}pp`.',
 f'- Late-rally mean position delta after 2020-10-01: `{float(late["position_delta_low_minus_return"].mean()):.6f}`; action-changed rows: `{int(late["action_changed"].sum())}`.',
 '',
 'Interpretation:',
 '- The training floor broadened the rebound representation instead of selectively fixing the late-2020 low-exposure rally state.',
 '- More direct floor targets are likely to repeat the same drift unless a narrower state feature or reward gate separates late rally from mid-year drawdown risk.',
 '',
 f"Next action: `{selection['next_mechanism']}`.",
]
SIDE_CARS['conclusion'].write_text('\n'.join(lines).rstrip()+'\n',encoding='utf-8')
print(json.dumps(json_safe(summary),indent=2,allow_nan=False)[:6000])


{
  "experiment": "wf2020_low_exposure_floor_representation_drift_v1",
  "created_at": "2026-06-14T09:38:57.522164+00:00",
  "decision": "low_exposure_floor_representation_drift_confirmed_no_promotion",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": "low_exposure_rally_floor_three_fold_checkpoint_metric_replay_v1",
  "reference_baseline": "soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05",
  "role": "diagnose",
  "changed_layer": [
    "model_selection",
    "training_target_composition",
    "representation"
  ],
  "target_window": "wf_2020",
  "promotion_allowed": false,
  "actor_retention_allowed": false,
  "training_allowed_from_this_card": false,
  "relative_return_delta_low_minus_return_pp": -3.3199772891998123,
  "max_dd_delta_low_minus_return_pp": 3.1066204649556575,
  "september_return_delta_pp": -3.328444657956409,
  "november_return_delta_pp": -3.422857676015066,
  "december_return_delta_pp": -3.1252842218786103,
  "rebo

## Conclusion

The low-exposure floor replay is a useful negative result. It shows broad target floors are not enough: the model needs a narrower late-rally state/reward distinction before another target revision or training loop.